# Crafter Tabula Rasa JEPA World Model: Google Colab Training Pipeline

This notebook implements the complete **Joint Embedding Predictive Architecture (JEPA)** training pipeline for Crafter.
- **Zero Cheat Policy & Planning**: Operates solely on raw visual RGB pixel observations (64x64x3) without accessing internal game state (`inventory`, `player_pos`, `semantic`, etc.).
- **Anti-Collapse Self-Supervision**: Employs **VICReg** (Invariance, Variance Hinge, Covariance Decorrelation) ensuring non-collapse without pixel decoders.
- **Intrinsic Epistemic Curiosity**: 4-head predictor ensemble disagreement drives exploration and discovery of causal affordances.
- **Two-Tier Persistent Memory**: 70% rolling FIFO + 30% Curated Milestone Archive.
- **Preemption-Resilient Checkpointing**: Saves checkpoints to Google Drive (`/content/drive/MyDrive/crafter_jepa/checkpoints`).

### Cell 1: Environment Verification & Hardware Diagnostics

In [ ]:
# Cell 1: Environment Verification & Hardware Diagnostics
import os
import sys
import platform
import torch

print("=" * 70)
print("  SYSTEM & HARDWARE DIAGNOSTICS")
print("=" * 70)
print(f"Python Version:   {platform.python_version()} ({sys.executable})")
print(f"OS Platform:      {platform.system()} {platform.release()}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version:     {torch.version.cuda}")
    print(f"Device Count:     {torch.cuda.device_count()}")
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    print(f"Total VRAM:       {vram_bytes / (1024**3):.2f} GB")
else:
    print("Running in CPU Mode (Hardware accelerator not detected).")
print("=" * 70)

### Cell 2: Google Drive Mounting & Persistent Checkpoint Storage

In [ ]:
# Cell 2: Google Drive Mounting & Namespace Isolation (Rule 1: Dynamic Auto-Creation)
import os
from pathlib import Path

try:
    from google.colab import drive
    print("[Colab Environment Detected] Mounting Google Drive...")
    drive.mount('/content/drive')
except ImportError:
    print("[Non-Colab / Local Environment] Running locally.")

# Target: /content/drive/MyDrive/learn-a-world/prometheus/checkpoints (or /content/drive/MyDrive/crafter_jepa/checkpoints)
if os.path.exists("/content/drive/MyDrive"):
    BASE_DRIVE = Path("/content/drive/MyDrive/learn-a-world")
    MODEL_SUBDIR = BASE_DRIVE / "prometheus"
    CHECKPOINT_DIR = MODEL_SUBDIR / "checkpoints"
    LOGS_DIR = MODEL_SUBDIR / "logs"
    SHARDS_DIR = MODEL_SUBDIR / "shards"
else:
    CHECKPOINT_DIR = Path("./checkpoints")
    LOGS_DIR = Path("./logs")
    SHARDS_DIR = Path("./shards")

# Rule 1: Dynamic Auto-Creation
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)
SHARDS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[+] Model checkpoint directory ready: {CHECKPOINT_DIR.resolve()}")
print(f"[+] Logs directory ready: {LOGS_DIR.resolve()}")
print(f"[+] Shards directory ready: {SHARDS_DIR.resolve()}")

### Cell 3: Pip Install Dependencies

In [ ]:
# Cell 3: Pip Install Dependencies
import sys
import subprocess

dependencies = [
    "crafter",
    "gymnasium>=0.28.0",
    "matplotlib>=3.5.0",
    "numpy>=1.21.0",
    "torch>=2.0.0",
]

print("Installing required dependencies...")
cmd = [sys.executable, "-m", "pip", "install", "--upgrade"] + dependencies
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode == 0:
    print("[+] All dependencies installed successfully.")
else:
    print(f"[!] Output during dependency installation:\n{result.stdout}\n{result.stderr}")

### Cell 4: Repository Setup & Path Configuration

In [ ]:
# Cell 4: Repository Setup & Path Configuration
import os
import sys
from pathlib import Path

# Locate and configure project root path
cwd = Path(os.getcwd())
if (cwd / "src").exists():
    REPO_ROOT = cwd
elif (cwd / "prometheus" / "src").exists():
    REPO_ROOT = cwd / "prometheus"
elif (cwd.parent / "src").exists():
    REPO_ROOT = cwd.parent
elif Path("/content/learn-a-world/prometheus").exists():
    REPO_ROOT = Path("/content/learn-a-world/prometheus")
elif Path("/content/repo/prometheus").exists():
    REPO_ROOT = Path("/content/repo/prometheus")
elif Path("/content/crafter_jepa").exists():
    REPO_ROOT = Path("/content/crafter_jepa")
else:
    REPO_ROOT = cwd

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"[+] Repository root configured: {REPO_ROOT.resolve()}")

# Verify core module imports
try:
    from src.env.wrapper import make_crafter_env
    from src.agent.agent import JEPAAgent
    from src.models.losses import VICRegLoss
    from src.memory.buffer import TwoTierReplayBuffer
    from src.memory.storage import CheckpointManager
    print("[+] Core JEPA modules imported successfully!")
except ImportError as err:
    print(f"[!] Import error: {err}")
    raise

### Cell 5: Fast CPU Smoke Test Verification (<60s)

In [ ]:
# Cell 5: Fast CPU Smoke Test Verification (<60s)
# Validates complete 13-stage JEPA loop (zero-cheat, VICReg, buffer, curiosity)
from src.test_smoke import run_smoke_test

print("Executing fast local CPU smoke test...")
smoke_results = run_smoke_test()
print(f"\n[+] CPU Smoke Test Passed! Elapsed: {smoke_results['total_elapsed_seconds']:.3f} s")

### Cell 6: Full GPU Training Run with Drive Checkpointing

In [ ]:
# Cell 6: Full GPU Training Run with Drive Checkpointing
import torch
from src.train import TrainingPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Starting Tabula Rasa JEPA Training on device: {device}")

training_config = {
    "device": device,
    "total_steps": 5_000,
    "batch_size": 64,
    "checkpoint_dir": CHECKPOINT_DIR,
    "eval_interval": 500,
    "eval_episodes": 3,
    "seed": 42,
    "lr": 1e-4,
    "critic_lr": 3e-4,
    "warmup_steps": 100,
    "save_interval": 500,
    "no_resume": False,
    "tier2_ratio": 0.30,
    "gamma": 0.99,
    "tau": 0.99,
    "log_interval": 100,
}

pipeline = TrainingPipeline(training_config)
train_results = pipeline.train()

print("\n[+] Training session completed successfully.")
print(f"Total Steps:        {train_results['total_steps']}")
print(f"Total Episodes:     {train_results['total_episodes']}")
print(f"Achievements Count: {len(train_results['achievements_unlocked'])}")
print(f"Achievements List:  {train_results['achievements_unlocked']}")
print(f"Final Checkpoint:   {train_results['final_checkpoint']}")

### Cell 7: Visualization and Evaluation
Plots Loss Curves (VICReg Invariance, Variance, Covariance), Curiosity Progression, and Crafter Achievement Unlock Curves.

In [ ]:
# Cell 7: Visualization and Evaluation
import matplotlib.pyplot as plt
import numpy as np

loss_history = pipeline.loss_history
achievement_history = pipeline.achievement_history

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Crafter Tabula Rasa JEPA Training & Evaluation Telemetry", fontsize=16, fontweight="bold")

# 1. Total World Model Loss
ax1 = axes[0, 0]
if loss_history:
    steps_arr = np.arange(len(loss_history))
    wm_losses = [m.get("wm_loss", 0.0) for m in loss_history]
    ax1.plot(steps_arr, wm_losses, label="World Model Loss", color="#1f77b4", alpha=0.8)
    ax1.set_title("World Model Total Loss (VICReg + Latent Variance)")
    ax1.set_xlabel("Update Step")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.5)
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "No loss history recorded", ha="center", va="center")

# 2. VICReg Loss Components (Invariance, Variance Hinge, Covariance)
ax2 = axes[0, 1]
if loss_history:
    inv_losses = [m.get("invariance", 0.0) for m in loss_history]
    var_losses = [m.get("variance", 0.0) for m in loss_history]
    cov_losses = [m.get("covariance", 0.0) for m in loss_history]
    ax2.plot(steps_arr, inv_losses, label="Invariance (Inv)", color="#2ca02c", alpha=0.8)
    ax2.plot(steps_arr, var_losses, label="Variance Hinge (Var)", color="#d62728", alpha=0.8)
    ax2.plot(steps_arr, cov_losses, label="Covariance (Cov)", color="#9467bd", alpha=0.8)
    ax2.set_title("VICReg Component Decomposition")
    ax2.set_xlabel("Update Step")
    ax2.set_ylabel("Component Loss")
    ax2.grid(True, linestyle="--", alpha=0.5)
    ax2.legend()
else:
    ax2.text(0.5, 0.5, "No loss history recorded", ha="center", va="center")

# 3. Policy & Value Telemetry (Critic Loss & Expected Q-value)
ax3 = axes[1, 0]
if loss_history:
    critic_losses = [m.get("critic_loss", 0.0) for m in loss_history]
    q_vals = [m.get("q_mean", 0.0) for m in loss_history]
    ax3.plot(steps_arr, critic_losses, label="Critic MSE Loss", color="#ff7f0e", alpha=0.8)
    ax3.plot(steps_arr, q_vals, label="Expected Q-Value", color="#17becf", alpha=0.8)
    ax3.set_title("Actor-Critic Optimization & Value Grounding")
    ax3.set_xlabel("Update Step")
    ax3.set_ylabel("Metric Value")
    ax3.grid(True, linestyle="--", alpha=0.5)
    ax3.legend()
else:
    ax3.text(0.5, 0.5, "No critic history recorded", ha="center", va="center")

# 4. Crafter Achievement Unlock Curve Over Time
ax4 = axes[1, 1]
if achievement_history:
    eval_steps = sorted(achievement_history.keys())
    ach_counts = [achievement_history[s]["achievement_count"] for s in eval_steps]
    returns = [achievement_history[s]["mean_return"] for s in eval_steps]
    
    color = "#e377c2"
    ax4.set_xlabel("Environment Steps")
    ax4.set_ylabel("Achievements Unlocked", color=color, fontweight="bold")
    line1 = ax4.plot(eval_steps, ach_counts, color=color, marker="o", linewidth=2, label="Unique Achievements")
    ax4.tick_params(axis="y", labelcolor=color)
    ax4.grid(True, linestyle="--", alpha=0.5)
    
    ax4_twin = ax4.twinx()
    color_ret = "#8c564b"
    ax4_twin.set_ylabel("Mean Return", color=color_ret, fontweight="bold")
    line2 = ax4_twin.plot(eval_steps, returns, color=color_ret, linestyle="--", marker="s", label="Episodic Return")
    ax4_twin.tick_params(axis="y", labelcolor=color_ret)
    
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax4.legend(lines, labels, loc="upper left")
    ax4.set_title("Crafter Achievement Progression & Return Curve")
else:
    ax4.text(0.5, 0.5, "No evaluation history recorded", ha="center", va="center")

plt.tight_layout()
plt.savefig("crafter_jepa_training_curves.png", dpi=300)
plt.show()
print("[+] Telemetry visualization generated and saved to crafter_jepa_training_curves.png")